In [1]:
import os
import random

import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, recall_score, precision_score, f1_score

import sys
for stream in (sys.stdout, sys.stderr):
	if not hasattr(stream, "reconfigure"):
		stream.reconfigure = lambda *args, **kwargs: None

import wandb
from torch import nn
from torch.utils.data import DataLoader

from Model.QCNN import QCNN
from Model.WDCNN import WDCNN
from utils.DatasetLoader import CustomTensorDataset
from utils.Preprocess import prepro
from utils.train_function import group_parameters

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_gpu = torch.cuda.is_available()

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

In [3]:
def random_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python random module.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [4]:
def select_model(chosen_model):
    if chosen_model == 'wdcnn':
        model = WDCNN()
    if chosen_model == 'qcnn':
        model = QCNN()
    return model

In [5]:
def train(config, dataloader):
    net = select_model(config.chosen_model)
    if use_gpu:
        net.cuda()
    wandb.watch(net, log="all")
    max_acc = 0.0
    train_loss = []
    valid_loss = []
    train_acc = []
    valid_acc = []
    for e in range(config.epochs):
        for phase in ['train', 'validation']:
            loss = 0
            total = 0
            correct = 0
            loss_total = 0

            if phase == 'train':
                net.train()
            if phase == 'validation':
                net.eval()
                torch.no_grad()

            for step, (x, y) in enumerate(dataloader[phase]):

                x = x.type(torch.float)
                y = y.type(torch.long)
                y = y.view(-1)
                if use_gpu:
                    x, y = x.cuda(), y.cuda()
                if config.chosen_model == 'qcnn':
                    group = group_parameters(net)
                    optimizer = torch.optim.SGD([
                        {"params": group[0], "lr": config.lr},  # weight_r
                        {"params": group[1], "lr": config.lr * config.alpha},  # weight_g
                        {"params": group[2], "lr": config.lr * config.alpha},  # weight_b
                        {"params": group[3], "lr": config.lr},  # bias_r
                        {"params": group[4], "lr": config.lr * config.alpha},  # bias_g
                        {"params": group[5], "lr": config.lr * config.alpha},  # bias_b
                        {"params": group[6], "lr": config.lr},
                        {"params": group[7], "lr": config.lr},
                    ], lr=config.lr, momentum=0.9, weight_decay=1e-4)
                    torch.nn.utils.clip_grad_norm_(net.parameters(), 0.01)

                    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs,
                                                                     eta_min=1e-8)  # goal: maximize Dice score


                else:
                    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs,
                                                                            eta_min=1e-8)  # goal: maximize Dice score
                    optimizer = torch.optim.SGD(net.parameters(), lr=config.lr, momentum=0.9)
                    
                loss_func = nn.CrossEntropyLoss()
                if use_gpu:
                    y_hat = net(x).cuda()
                else:
                    y_hat = net(x)
                loss = loss_func(y_hat, y)
                if phase == 'train':
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                loss_total += loss.item()
                y_predict = y_hat.argmax(dim=1)

                total += y.size(0)
                if use_gpu:
                    correct += (y_predict == y).cpu().squeeze().sum().numpy()
                else:
                    correct += (y_predict == y).squeeze().sum().numpy()

                if step % 20 == 0 and phase == 'train':
                    print('Epoch:%d, Step [%d/%d], Loss: %.4f'
                          % (
                          e + 1, step + 1, len(dataloader[phase].dataset), loss_total))
            # loss_total = loss_total / len(dataloader[phase].dataset)

            acc = correct / total
            if phase == 'train':
                train_loss.append(loss_total)
                train_acc.append(acc)
                wandb.log({
                    "Train Accuracy": 100. * acc,
                    "Train Loss": loss_total})
            if phase == 'validation':
                scheduler.step(loss_total)
                valid_loss.append(loss_total)
                valid_acc.append(acc)
                wandb.log({
                    "Validation Accuracy": 100. * acc,
                    "Validation Loss": loss_total})
                if acc >= max_acc:
                    max_acc = acc
                    if not os.path.exists('Models'):
                        os.mkdir('Models')
                    torch.save(net.state_dict(), f'Models/best_checkpoint.pth')
                    print('saved')
            print('%s ACC:%.4f' % (phase, acc))
    return net, train_loss, valid_loss, train_acc, valid_acc

In [21]:
def inference(config, dataloader):
    net = select_model(config.chosen_model)
    state_dict = torch.load('Models/best_checkpoint.pth', weights_only=True)
    net.load_state_dict(state_dict)
    y_list, y_predict_list = [], []
    if use_gpu:
        net.cuda()
    net.eval()
    # endregion
    with torch.no_grad():
        for step, (x, y) in enumerate(dataloader):
            x = x.type(torch.float)
            y = y.type(torch.long)
            y = y.view(-1)
            if use_gpu:
                x, y = x.cuda(), y.cuda()
            y_hat = net(x)
            y_predict = y_hat.argmax(dim=1)
            y_list.extend(y.detach().cpu().numpy())
            y_predict_list.extend(y_predict.detach().cpu().numpy())

        cnf_matrix = confusion_matrix(y_list, y_predict_list)
        recall = recall_score(y_list, y_predict_list, average="macro")
        precision = precision_score(y_list, y_predict_list, average="macro")

        F1 = f1_score(y_list, y_predict_list, average="macro")
        FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
        FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
        TP = np.diag(cnf_matrix)
        TN = cnf_matrix.sum() - (FP + FN + TP)

        FP = FP.astype(float)
        TN = TN.astype(float)
        FPR = np.mean(FP / (FP + TN))
        wandb.log({
            "F1 Score": F1,
            "FPR": FPR,
            "Recall": recall,
            'PRE': precision})

        torch.save(net.state_dict(), os.path.join(wandb.run.dir, "checkpoint.pth"))
        wandb.save('*.pth')
        print('model saved')
        return F1, y_list, y_predict_list, precision, recall, FPR

In [ ]:
def main(config):
    use_cuda = not config.no_cuda and torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    random_seed(config.seed)

    path = os.path.join('data', config.chosen_data)

    # train set, number denotes each category has 750 samples
    train_X, train_Y, valid_X, valid_Y = prepro(d_path=path,
                                                 length=2048,
                                                 number=750,
                                                 normal=False,
                                                 enc=True,
                                                 enc_step=28,
                                                 snr=config.snr,
                                                 property='Train',
                                                 noise=config.add_noise
                                                 )

    # test set, number denotes each category has 250 samples
    test_X, test_Y = prepro(d_path=path,
                             length=2048,
                             number=250,
                             normal=False,
                             enc=True,
                             enc_step=28,
                             snr=config.snr,
                             property='Test',
                             noise=config.add_noise
                             )

    train_X, valid_X, test_X = train_X[:, np.newaxis, :], valid_X[:, np.newaxis, :], test_X[:, np.newaxis, :]

    train_dataset = CustomTensorDataset(torch.tensor(train_X, dtype=torch.float), torch.tensor(train_Y))
    valid_dataset = CustomTensorDataset(torch.tensor(valid_X, dtype=torch.float), torch.tensor(valid_Y))
    test_dataset = CustomTensorDataset(torch.tensor(test_X, dtype=torch.float), torch.tensor(test_Y))

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)
    data_loaders = {
        "train": train_loader,
        "validation": valid_loader
    }
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=False)
    # train(config, data_loaders)
    # inference(config, test_loader)
    
    net, train_loss, valid_loss, train_acc, valid_acc = train(config, data_loaders)

    F1, y_list, y_predict_list, precision, recall, FPR = inference(config, test_loader)
    

    generate_all_plots(train_loss, valid_loss, train_acc, valid_acc, y_list, y_predict_list, 
                      [precision, recall, F1, FPR], train_X, config.epochs)

In [ ]:
if __name__ == '__main__':
    # wandb initialization, you need to create a wandb account and enter the username in 'entity'
    wandb.init(project="bearinganomaly", entity="alinaderi119-upwork", mode="disabled")
    # WandB – Config is a variable that holds and saves hypermarkets and inputs
    config = wandb.config  # Initialize config
    config.no_cuda = False  # disables CUDA training
    config.log_interval = 200  # how many batches to wait before logging training status
    config.seed = 42  # random seed (default: 42)

    # Hyperparameters, lr and alpha need to fine-tune
    config.batch_size = 64  # input batch size for training (default: 64)
    config.epochs = 50  # number of epochs to train (default: 10)
    config.lr = 0.5  # learning rate (default: 0.5)
    config.alpha = 0.03 # scale factor alpha

    # noisy condition
    config.add_noise = False
    config.snr = -6

    # dataset and model
    config.chosen_data = '0HP'
    config.chosen_model = 'qcnn'  # wdcnn or qcnn
    main(config)

Epoch:1, Step [1/5000], Loss: 2.3078
Epoch:1, Step [21/5000], Loss: 45.3322
Epoch:1, Step [41/5000], Loss: 84.7552
Epoch:1, Step [61/5000], Loss: 123.0264
train ACC:0.4794


C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/is

saved
validation ACC:0.6002
Epoch:2, Step [1/5000], Loss: 1.8572
Epoch:2, Step [21/5000], Loss: 37.5685
Epoch:2, Step [41/5000], Loss: 72.9706
Epoch:2, Step [61/5000], Loss: 107.1993
train ACC:0.7394
saved
validation ACC:0.7344
Epoch:3, Step [1/5000], Loss: 1.6773
Epoch:3, Step [21/5000], Loss: 33.6648
Epoch:3, Step [41/5000], Loss: 66.6453
Epoch:3, Step [61/5000], Loss: 96.8898
train ACC:0.9091
saved
validation ACC:0.9960
Epoch:4, Step [1/5000], Loss: 1.4893
Epoch:4, Step [21/5000], Loss: 30.9312
Epoch:4, Step [41/5000], Loss: 60.6417
Epoch:4, Step [61/5000], Loss: 90.3123
train ACC:0.9904
saved
validation ACC:1.0000
Epoch:5, Step [1/5000], Loss: 1.4627
Epoch:5, Step [21/5000], Loss: 30.7799
Epoch:5, Step [41/5000], Loss: 60.0496
Epoch:5, Step [61/5000], Loss: 89.3159
train ACC:0.9998
saved
validation ACC:1.0000
Epoch:6, Step [1/5000], Loss: 1.4631
Epoch:6, Step [21/5000], Loss: 30.9108
Epoch:6, Step [41/5000], Loss: 60.1700
Epoch:6, Step [61/5000], Loss: 89.4629
train ACC:0.9986
save

C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/is

saved
validation ACC:0.6987
Epoch:2, Step [1/5000], Loss: 1.8228
Epoch:2, Step [21/5000], Loss: 37.1837
Epoch:2, Step [41/5000], Loss: 70.9179
Epoch:2, Step [61/5000], Loss: 104.0191
train ACC:0.7975
saved
validation ACC:0.9619
Epoch:3, Step [1/5000], Loss: 1.4907
Epoch:3, Step [21/5000], Loss: 31.0948
Epoch:3, Step [41/5000], Loss: 61.0267
Epoch:3, Step [61/5000], Loss: 90.3911
train ACC:0.9920
saved
validation ACC:1.0000
Epoch:4, Step [1/5000], Loss: 1.4654
Epoch:4, Step [21/5000], Loss: 30.7818
Epoch:4, Step [41/5000], Loss: 60.1335
Epoch:4, Step [61/5000], Loss: 89.4396
train ACC:0.9988
saved
validation ACC:1.0000
Epoch:5, Step [1/5000], Loss: 1.4643
Epoch:5, Step [21/5000], Loss: 30.9057
Epoch:5, Step [41/5000], Loss: 60.1948
Epoch:5, Step [61/5000], Loss: 89.4875
train ACC:0.9974
saved
validation ACC:1.0000
Epoch:6, Step [1/5000], Loss: 1.4625
Epoch:6, Step [21/5000], Loss: 31.9764
Epoch:6, Step [41/5000], Loss: 62.4099
Epoch:6, Step [61/5000], Loss: 92.2050
train ACC:0.9629
save

In [25]:
plt.style.use('seaborn-v0_8')
sns.set_context("paper", font_scale=1.5)

In [33]:
def plot_learning_curves(train_loss, valid_loss, train_acc, valid_acc, epochs, save_path="plots"):
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    epochs_range = range(1, epochs + 1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_range, train_loss, label='Train Loss', linewidth=2, marker='o')
    plt.plot(epochs_range, valid_loss, label='Validation Loss', linewidth=2, marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'learning_curve_loss.png'), dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(epochs_range, train_acc, label='Train Accuracy', linewidth=2, marker='o')
    plt.plot(epochs_range, valid_acc, label='Validation Accuracy', linewidth=2, marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'learning_curve_accuracy.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [34]:
def plot_confusion_matrix(y_true, y_pred, save_path="plots"):
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 12})
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [35]:
def plot_performance_metrics(metrics, save_path="plots"):
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    metric_names = ['Precision', 'Recall', 'F1-Score', 'FPR']
    plt.figure(figsize=(10, 6))
    bars = plt.bar(metric_names, metrics, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    plt.ylabel('Score')
    plt.title('Performance Metrics')
    plt.ylim(0, 1)
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.3f}', ha='center', va='bottom')
    
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'performance_metrics.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [36]:
def plot_signal_distribution(data, title='Signal Distribution', save_path="plots"):
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    plt.figure(figsize=(10, 6))
    plt.hist(data.flatten(), bins=50, color='skyblue', edgecolor='black')
    plt.xlabel('Signal Value')
    plt.ylabel('Frequency')
    plt.title(title)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'{title.lower().replace(" ", "_")}.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [38]:
from matplotlib.patches import Rectangle, Arrow

def plot_qcnn_architecture(save_path="plots"):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 6)
    ax.axis('off')

    # Raw Signal
    ax.add_patch(Rectangle((0, 2.5), 1, 1, fill=False))
    ax.text(0.2, 2.7, 'Raw Signal\n1x2048', fontsize=10)

    # Arrows
    ax.add_patch(Arrow(1, 3, 1, 0, width=0.5, color='black'))

    # Layers
    layers = ['Layer1\nQConv 16@64x1', 'Layer2\nQConv 32@3x1', 'Layers3-6\nQConv 64@3x1', 'FC(192,100)', 'FC(100,10)\nSoftMax']
    for i, layer in enumerate(layers):
        ax.add_patch(Rectangle((2 + i*2, 2.5), 1.5, 1, fill=False))
        ax.text(2 + i*2 + 0.1, 2.6, layer, fontsize=10)
        if i < len(layers)-1:
            ax.add_patch(Arrow(3.5 + i*2, 3, 0.5, 0, width=0.5, color='black'))

    # Qttention Maps
    ax.text(2, 1.5, 'Qttention Map 1', fontsize=10)
    ax.text(4, 1.5, 'Qttention Map 2', fontsize=10)
    ax.text(6, 1.5, 'Qttention Map 3', fontsize=10)

    # Fault Classification
    ax.text(10, 2.7, 'Fault\nClassification', fontsize=10)

    plt.savefig(os.path.join(save_path, 'qcnn_architecture.png'), dpi=300)
    plt.close()

In [ ]:
# def extract_features(net, loader, device):
#     features = []
#     labels = []
#     net.eval()
#     with torch.no_grad():
#         for x, y in loader:
#             x = x.type(torch.float).to(device)
#             # Use the convolutional layers to extract features
#             feature = net.cnn(x)
#             # Flatten the output for t-SNE
#             features.append(feature.cpu().numpy().flatten())
#             labels.append(y.numpy())
#     return np.array(features), np.array(labels)

In [ ]:
from sklearn.manifold import TSNE

def extract_features(net, loader, device):
    features = []
    labels = []
    net.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.type(torch.float).to(device)
            feature = net.features(x) 
            features.append(feature.cpu().numpy().flatten())
            labels.append(y.numpy())
    return np.array(features), np.array(labels)

def plot_tsne(net, loader, device, save_path="plots", model_name="QCNN"):
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    features, labels = extract_features(net, loader, device)

    tsne = TSNE(n_components=2, random_state=42)
    tsne_features = tsne.fit_transform(features)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(tsne_features[:, 0], tsne_features[:, 1], c=labels, cmap='tab10')
    plt.colorbar(scatter)
    plt.title(f't-SNE of {model_name} Features')
    plt.savefig(os.path.join(save_path, f'tsne_{model_name.lower()}.png'), dpi=300)
    plt.close()

In [44]:
def generate_all_plots(train_loss, valid_loss, train_acc, valid_acc, y_true, y_pred, metrics, train_X, epochs, net, test_loader, device):
    plot_learning_curves(train_loss, valid_loss, train_acc, valid_acc, epochs)
    plot_confusion_matrix(y_true, y_pred)
    plot_performance_metrics(metrics)
    plot_signal_distribution(train_X, title='Train Signal Distribution')
    plot_tsne(net, test_loader, device, model_name="QCNN")
    plot_qcnn_architecture()

In [ ]:
def main(config):
    use_cuda = not config.no_cuda and torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    random_seed(config.seed)

    path = os.path.join('data', config.chosen_data)

    # train set, number denotes each category has 750 samples
    train_X, train_Y, valid_X, valid_Y = prepro(d_path=path,
                                                 length=2048,
                                                 number=750,
                                                 normal=False,
                                                 enc=True,
                                                 enc_step=28,
                                                 snr=config.snr,
                                                 property='Train',
                                                 noise=config.add_noise
                                                 )

    # test set, number denotes each category has 250 samples
    test_X, test_Y = prepro(d_path=path,
                             length=2048,
                             number=250,
                             normal=False,
                             enc=True,
                             enc_step=28,
                             snr=config.snr,
                             property='Test',
                             noise=config.add_noise
                             )

    train_X, valid_X, test_X = train_X[:, np.newaxis, :], valid_X[:, np.newaxis, :], test_X[:, np.newaxis, :]

    train_dataset = CustomTensorDataset(torch.tensor(train_X, dtype=torch.float), torch.tensor(train_Y))
    valid_dataset = CustomTensorDataset(torch.tensor(valid_X, dtype=torch.float), torch.tensor(valid_Y))
    test_dataset = CustomTensorDataset(torch.tensor(test_X, dtype=torch.float), torch.tensor(test_Y))

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)
    data_loaders = {
        "train": train_loader,
        "validation": valid_loader
    }
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=False)
    # train(config, data_loaders)
    # inference(config, test_loader)
    
    net, train_loss, valid_loss, train_acc, valid_acc = train(config, data_loaders)

    F1, y_list, y_predict_list, precision, recall, FPR = inference(config, test_loader)
    

    generate_all_plots(train_loss, valid_loss, train_acc, valid_acc, y_list, y_predict_list, 
                      [precision, recall, F1, FPR], train_X, config.epochs, net, test_loader, device)


if __name__ == '__main__':
    # wandb initialization, you need to create a wandb account and enter the username in 'entity'
    wandb.init(project="bearinganomaly", entity="alinaderi119-upwork", mode="disabled")
    # WandB – Config is a variable that holds and saves hypermarkets and inputs
    config = wandb.config  # Initialize config
    config.no_cuda = False  # disables CUDA training
    config.log_interval = 200  # how many batches to wait before logging training status
    config.seed = 42  # random seed (default: 42)

    # Hyperparameters, lr and alpha need to fine-tune
    config.batch_size = 64  # input batch size for training (default: 64)
    config.epochs = 50  # number of epochs to train (default: 10)
    config.lr = 0.5  # learning rate (default: 0.5)
    config.alpha = 0.03 # scale factor alpha

    # noisy condition
    config.add_noise = False
    config.snr = -6

    # dataset and model
    config.chosen_data = '0HP'
    config.chosen_model = 'qcnn'  # wdcnn or qcnn
    main(config)
    

Epoch:1, Step [1/5000], Loss: 2.3078
Epoch:1, Step [21/5000], Loss: 45.3322
Epoch:1, Step [41/5000], Loss: 84.7552
Epoch:1, Step [61/5000], Loss: 123.0264
train ACC:0.4794


C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/is

saved
validation ACC:0.6002
Epoch:2, Step [1/5000], Loss: 1.8572
Epoch:2, Step [21/5000], Loss: 37.5685
Epoch:2, Step [41/5000], Loss: 72.9706
Epoch:2, Step [61/5000], Loss: 107.1993
train ACC:0.7394
saved
validation ACC:0.7344
Epoch:3, Step [1/5000], Loss: 1.6773
Epoch:3, Step [21/5000], Loss: 33.6648
Epoch:3, Step [41/5000], Loss: 66.6453
Epoch:3, Step [61/5000], Loss: 96.8898
train ACC:0.9091
saved
validation ACC:0.9960
Epoch:4, Step [1/5000], Loss: 1.4893
Epoch:4, Step [21/5000], Loss: 30.9312
Epoch:4, Step [41/5000], Loss: 60.6417
Epoch:4, Step [61/5000], Loss: 90.3123
train ACC:0.9904
saved
validation ACC:1.0000
Epoch:5, Step [1/5000], Loss: 1.4627
Epoch:5, Step [21/5000], Loss: 30.7799
Epoch:5, Step [41/5000], Loss: 60.0496
Epoch:5, Step [61/5000], Loss: 89.3159
train ACC:0.9998
saved
validation ACC:1.0000
Epoch:6, Step [1/5000], Loss: 1.4631
Epoch:6, Step [21/5000], Loss: 30.9108
Epoch:6, Step [41/5000], Loss: 60.1700
Epoch:6, Step [61/5000], Loss: 89.4629
train ACC:0.9986
save

AttributeError: 'QCNN' object has no attribute 'features'

In [ ]:
# Prepare data loaders as in main(config)
path = os.path.join('data', config.chosen_data)
train_X, train_Y, valid_X, valid_Y = prepro(d_path=path,
											length=2048,
											number=750,
											normal=False,
											enc=True,
											enc_step=28,
											snr=config.snr,
											property='Train',
											noise=config.add_noise)
test_X, test_Y = prepro(d_path=path,
						length=2048,
						number=250,
						normal=False,
						enc=True,
						enc_step=28,
						snr=config.snr,
						property='Test',
						noise=config.add_noise)
train_X, valid_X, test_X = train_X[:, np.newaxis, :], valid_X[:, np.newaxis, :], test_X[:, np.newaxis, :]

train_dataset = CustomTensorDataset(torch.tensor(train_X, dtype=torch.float), torch.tensor(train_Y))
valid_dataset = CustomTensorDataset(torch.tensor(valid_X, dtype=torch.float), torch.tensor(valid_Y))
test_dataset = CustomTensorDataset(torch.tensor(test_X, dtype=torch.float), torch.tensor(test_Y))

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)
data_loaders = {
	"train": train_loader,
	"validation": valid_loader
}
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=False)

# Train and get results
net, train_loss, valid_loss, train_acc, valid_acc = train(config, data_loaders)
F1, y_list, y_predict_list, precision, recall, FPR = inference(config, test_loader)

# Generate plots
generate_all_plots(train_loss, valid_loss, train_acc, valid_acc, y_list, y_predict_list, 
                   [precision, recall, F1, FPR], train_X, config.epochs, net, test_loader, device)




net, train_loss, valid_loss, train_acc, valid_acc = train(config, data_loaders)

F1, y_list, y_predict_list, precision, recall, FPR = inference(config, test_loader)
    

generate_all_plots(train_loss, valid_loss, train_acc, valid_acc, y_list, y_predict_list, 
                    [precision, recall, F1, FPR], train_X, config.epochs, net, test_loader, device)

Epoch:1, Step [1/5000], Loss: 2.3030
Epoch:1, Step [21/5000], Loss: 45.6969
Epoch:1, Step [41/5000], Loss: 86.9126
Epoch:1, Step [61/5000], Loss: 125.6408
train ACC:0.4501


C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
C:\Users\alina\AppData\Roaming\Python\Python312\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/is

saved
validation ACC:0.6498
Epoch:2, Step [1/5000], Loss: 1.7177
Epoch:2, Step [21/5000], Loss: 35.7842
Epoch:2, Step [41/5000], Loss: 68.9490
Epoch:2, Step [61/5000], Loss: 102.0059
train ACC:0.8353
saved
validation ACC:0.8918
Epoch:3, Step [1/5000], Loss: 1.5110
Epoch:3, Step [21/5000], Loss: 32.5145
Epoch:3, Step [41/5000], Loss: 62.9425
Epoch:3, Step [61/5000], Loss: 93.5506
train ACC:0.9493
saved
validation ACC:0.9992
Epoch:4, Step [1/5000], Loss: 1.4653
Epoch:4, Step [21/5000], Loss: 31.7207
Epoch:4, Step [41/5000], Loss: 61.5053
Epoch:4, Step [61/5000], Loss: 90.8985
train ACC:0.9804
saved
validation ACC:1.0000
Epoch:5, Step [1/5000], Loss: 1.4662
Epoch:5, Step [21/5000], Loss: 30.8127
Epoch:5, Step [41/5000], Loss: 61.3775
Epoch:5, Step [61/5000], Loss: 92.6065
train ACC:0.9383
validation ACC:0.9976
Epoch:6, Step [1/5000], Loss: 1.4649
Epoch:6, Step [21/5000], Loss: 31.7901
Epoch:6, Step [41/5000], Loss: 61.1235
Epoch:6, Step [61/5000], Loss: 90.7306
train ACC:0.9812
validation

In [ ]:
def extract_features(net, loader, device):
    features = []
    labels = []
    net.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.type(torch.float).to(device)
            # Use the convolutional layers to extract features
            feature = net.cnn(x)
            # Flatten the output for t-SNE
            features.append(feature.cpu().numpy().flatten())
            labels.append(y.numpy())
    return np.array(features), np.array(labels)